# 第 1 周末解答

为了展示你对 OpenAI API 以及 Ollama 的熟悉程度，请构建一个工具：接收技术问题，  
并回复解释。这是一个你在课程期间自己也能使用的工具！

第 2 周之后，你将能为这个工具添加用户界面，从而得到一个有价值的应用。

In [ ]:
# 导入

# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：在 Jupyter 笔记本里漂亮地显示 Markdown/图片等
from IPython.display import Markdown, display, update_display
# 从 openai 导入 OpenAI 客户端类：用它调用 Chat Completions 等 API（Application Programming Interface）
from openai import OpenAI
import ollama

In [ ]:
# 常量

# 云端 GPT 模型名称（model id，经 OpenAI API 调用）
MODEL_GPT = 'gpt-4o-mini'
# 本地 Llama 模型名称（通常经 Ollama 调用）
MODEL_LLAMA = 'llama3.2'

In [ ]:
# 设置环境

# 加载 .env 文件：把 API Key 等密钥读入进程环境（override=True 表示覆盖已有同名变量）
load_dotenv()
# 创建 OpenAI 客户端；不传参时默认读环境变量里的 API Key
openai = OpenAI()


In [ ]:
# 这里是问题；覆盖输入以询问新问题

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [ ]:
# 提示词

# 系统提示词（system prompt）：给模型设定角色与规则，通常用户看不到
system_prompt = "You are a helpful technical tutor who answers questions about python code, software engineering, data science and LLMs"
# 用户提示词（user prompt）：本次要模型完成的具体任务与输入内容
user_prompt = "Please give a detailed explanation to the following question: " + question

In [ ]:
# 消息

# 组装 messages 列表：Chat Completions API 要求的对话格式（system / user / assistant）
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

In [ ]:
# 让 gpt-4o-mini 回答，使用流式输出

stream = openai.chat.completions.create(model=MODEL_GPT, messages=messages,stream=True)
    
response = ""
# 用 Markdown 在笔记本中渲染格式化文本（display_id 方便后续原地刷新）
display_handle = display(Markdown(""), display_id=True)
# 按流式 chunk（数据块）拼接文本；delta.content 是本次新增的一小段字
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    response = response.replace("```","").replace("markdown", "")
    # 原地更新同一输出区，实现「打字机」式流式刷新
    update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
# 让 Llama 3.2 回答

response = ollama.chat(model=MODEL_LLAMA, messages=messages)
reply = response['message']['content']
# 用 Markdown 在笔记本中渲染格式化文本（display_id 方便后续原地刷新）
display(Markdown(reply))

# 恭喜！

你可以通过使用以下方式接收问题来改进它：  
`my_question = input("Please enter your question:")`

然后交互式地创建提示词并发起调用。